<a href="https://colab.research.google.com/github/islam37/S-SLTE-Thesis/blob/main/notebooks/03_pilot_noise_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import json
import hashlib
import math
from pathlib import Path

import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")

# Load the frozen experiment manifest
with open(
    BASE / "configs/llm_pilot_manifest_v2.json",
    encoding="utf-8"
) as f:
    manifest = json.load(f)

revision = manifest["model_revision_sha"]

# Verify the exact pilot dataset
pilot_path = BASE / "data/pilot/pilot_500.csv"

assert hashlib.sha256(
    pilot_path.read_bytes()
).hexdigest() == manifest["pilot_sha256"]

pilot = pd.read_csv(pilot_path)

# Load frozen LLM annotations
annotation_path = (
    BASE / "outputs/llm_pilot"
    / f"pinned_{revision[:12]}"
    / "pilot_annotations.csv"
)

annotations = pd.read_csv(annotation_path)

assert len(pilot) == len(annotations) == 500
assert annotations["record_id"].is_unique
assert annotations["is_valid"].eq(True).all()
assert annotations["llm_label"].notna().all()
assert set(annotations["record_id"]) == set(pilot["record_id"])
assert annotations["llm_label"].isin(range(4)).all()

# PRIVATE evaluation reference:
# Never pass this file into the LLM or correction pipeline
reference_path = (
    BASE / "private_reference/noisy_reference.csv"
)

reference = pd.read_csv(reference_path)

assert {"record_id", "text", "label"}.issubset(
    reference.columns
)
assert reference["record_id"].is_unique
assert reference["label"].isin(range(4)).all()

# Verify original text matches by record_id
text_check = pilot.merge(
    reference[["record_id", "text"]],
    on="record_id",
    how="left",
    validate="one_to_one",
    suffixes=("_pilot", "_reference")
)

assert len(text_check) == 500
assert (
    text_check["text_pilot"] ==
    text_check["text_reference"]
).all(), "Reference text mismatch"

# Retrospective comparison only
evaluation = annotations.merge(
    reference[["record_id", "label"]],
    on="record_id",
    how="left",
    validate="one_to_one"
)

assert len(evaluation) == 500
assert evaluation["label"].notna().all()

evaluation["disagreement"] = (
    evaluation["llm_label"] != evaluation["label"]
)

n = len(evaluation)
errors = int(evaluation["disagreement"].sum())
matches = n - errors

noise_rate = errors / n

# Wilson 95% confidence interval
z = 1.96
denominator = 1 + z**2 / n

center = (
    noise_rate + z**2 / (2 * n)
) / denominator

margin = (
    z
    * math.sqrt(
        noise_rate * (1 - noise_rate) / n
        + z**2 / (4 * n**2)
    )
    / denominator
)

ci_low = max(0, center - margin)
ci_high = min(1, center + margin)

LABEL_MAP = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

# True vs predicted label confusion matrix
confusion = pd.crosstab(
    evaluation["label"],
    evaluation["llm_label"]
).reindex(
    index=range(4),
    columns=range(4),
    fill_value=0
)

confusion.index = [
    LABEL_MAP[x] for x in range(4)
]
confusion.columns = [
    LABEL_MAP[x] for x in range(4)
]

# Save aggregate results in the private evaluation area
EVAL_DIR = (
    BASE / "private_reference/evaluation/pilot_v1"
)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

summary = {
    "model_revision_sha": revision,
    "pilot_sha256": manifest["pilot_sha256"],
    "total": n,
    "matched_reference": matches,
    "disagreed_with_reference": errors,
    "disagreement_rate": noise_rate,
    "wilson_95_ci": [ci_low, ci_high],
    "interpretation": (
        "Exploratory disagreement against AG News "
        "reference labels, not manually verified error."
    )
}

summary_path = EVAL_DIR / "noise_summary.json"

if summary_path.exists():
    with open(summary_path, encoding="utf-8") as f:
        existing = json.load(f)

    assert existing == summary, (
        "Existing evaluation differs; review manually."
    )
else:
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

print("PILOT NOISE EVALUATION COMPLETED")
print("--------------------------------")
print("Total Records:", n)
print("Matches:", matches)
print("Disagreements:", errors)
print(f"Disagreement Rate: {noise_rate:.2%}")
print(f"Wilson 95% CI: [{ci_low:.2%}, {ci_high:.2%}]")

print("\nCONFUSION MATRIX")
print("Rows = AG News Reference Labels")
print("Columns = LLM Predictions")
display(confusion)

print("\nPrivate summary saved:", summary_path)


Mounted at /content/drive
PILOT NOISE EVALUATION COMPLETED
--------------------------------
Total Records: 500
Matches: 391
Disagreements: 109
Disagreement Rate: 21.80%
Wilson 95% CI: [18.40%, 25.63%]

CONFUSION MATRIX
Rows = AG News Reference Labels
Columns = LLM Predictions


,World,Sports,Business,Sci/Tech
World,109,7,4,0
Sports,5,102,1,0
Business,19,2,118,4
Sci/Tech,23,6,38,62



Private summary saved: /content/drive/MyDrive/S_SLTE_Thesis/private_reference/evaluation/pilot_v1/noise_summary.json
